# 07 - Reduce for modeling (departure-safe)

Builds the primary modeling dataset: one row per run, every feature anchored to `sched_origin_sec`
(scheduled departure). No information about the final lateness value is used anywhere.

Reads `2_df_gtfs_linked.parquet` directly (~20M rows, per-OTP-obs schedule data from `02b_gtfs_merge.ipynb`). `run_id` assignment, terminal lateness, and schedule extraction are combined in one per-year loop over the row-level data.

In [1]:
import pandas as pd
import numpy as np
import gc
from zoneinfo import ZoneInfo

from utils import line_key

BASEPATH = "../data"
eastern = ZoneInfo("America/New_York")

RUN_GAP_THRESHOLD_SEC = 90 * 60  # 90-minute new-run threshold
RUN_COLS = ["service_date", "train_number", "run_id"]

## 1. Run_id assignment + terminal lateness + schedule identity (combined, one pass)

Identifying individual train runs.
A new run starts on a 90-minute gap in the SEPTA-3am-service-day-adjusted timestamp, **or** whenever `line` changes, even with zero gap (e.g. one train running Airport -> Suburban -> Warminster outbound via Center City tunnel).

For each `run_id`: the outcome (`lateness`) comes from the run's last (chronologically) observation/lateness update. Every schedule/identity column is constant within a run (`line`, `trip_id`, `direction_id`, `source_gtfs_date`,
`sched_origin_sec`, `sched_terminus_sec`, `sched_duration_sec`, `n_scheduled_stops`, `origin_stop_id`, `terminus_stop_id`); these are pulled from the first (chronologically) observation within a run (though it shouldn't matter since theyre static).

In [2]:
READ_COLS = [
    "service_date", "time", "train_number", "lateness", "datetime",
    "line", "trip_id", "direction_id", "source_gtfs_date",
    "sched_origin_sec", "sched_terminus_sec", "sched_duration_sec",
    "n_scheduled_stops", "origin_stop_id", "terminus_stop_id",
    "period_flag",
]

years = sorted(
    pd.read_parquet(f"{BASEPATH}/2_df_gtfs_linked.parquet", columns = ["service_date"])
    ["service_date"].dt.year.unique()
)
print(f"years to process: {years}")

years to process: [np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]


In [3]:
terminal_all_parts = []

for year in years:
    chunk = pd.read_parquet(
        f"{BASEPATH}/2_df_gtfs_linked.parquet",
        columns = READ_COLS,
        filters = [("service_date", ">=", pd.Timestamp(f"{year}-01-01")),
                   ("service_date", "<", pd.Timestamp(f"{year + 1}-01-01"))],
    )
 
    chunk["train_number"] = chunk["train_number"].astype("int32")
    chunk["time"] = pd.to_timedelta(chunk["time"]).dt.total_seconds().astype("int32")

    # SEPTA 3am-service-day adjustment
    chunk["_time_adj"] = (
        chunk["datetime"] - chunk["service_date"].dt.tz_localize(chunk["datetime"].dt.tz)
    ).dt.total_seconds()

    chunk = chunk.sort_values(["service_date", "train_number", "_time_adj"]).reset_index(drop = True)

    # run_id: gap-based split, plus forced split on line change for multi-run trains
    chunk["_gap_from_prev"] = chunk.groupby(["service_date", "train_number"])["_time_adj"].diff()
    chunk["_line_prev"] = chunk.groupby(["service_date", "train_number"])["line"].shift(1)
    chunk["_line_changed"] = (
        chunk["line"].notna() & chunk["_line_prev"].notna() & (chunk["line"] != chunk["_line_prev"])
    )
    chunk["_new_run_flag"] = (
        chunk["_gap_from_prev"].isna()
        | (chunk["_gap_from_prev"] > RUN_GAP_THRESHOLD_SEC)
        | chunk["_line_changed"]
    )
    chunk["run_id"] = chunk.groupby(["service_date", "train_number"])["_new_run_flag"].cumsum()

    # terminal lateness (last obs per run) + 
    # schedule identity (first obs per run) aggregation
    year_terminal = chunk.groupby(RUN_COLS).agg(
        # will be used for binary target
        lateness = ("lateness", "last"),
        # spread of lateness readings across a run's observations
        # na for one-observation runs (volatility)
        # will be lagged
        lateness_std = ("lateness", "std"), 
        # avg lateness across a run's observations
        # will be lagged
        lateness_mean_run = ("lateness", "mean"),
        line = ("line", "first"),
        trip_id = ("trip_id", "first"),
        direction_id = ("direction_id", "first"),
        source_gtfs_date = ("source_gtfs_date", "first"),
        sched_origin_sec = ("sched_origin_sec", "first"),
        sched_terminus_sec = ("sched_terminus_sec", "first"),
        sched_duration_sec = ("sched_duration_sec", "first"),
        n_scheduled_stops = ("n_scheduled_stops", "first"),
        origin_stop_id = ("origin_stop_id", "first"),
        terminus_stop_id = ("terminus_stop_id", "first"),
        coverage_flag = ("period_flag", "first"),
    ).reset_index()

    terminal_all_parts.append(year_terminal)
    print(f"{year}: {len(chunk):,} pings -> {len(year_terminal):,} runs")

    del chunk, year_terminal
    gc.collect()

terminal_all = pd.concat(terminal_all_parts, ignore_index = True)
del terminal_all_parts
gc.collect()

# calendar breakdown of service_date 
# needed later for lag/trend groupings and calendar merge
terminal_all["year"] = terminal_all["service_date"].dt.year
terminal_all["month"] = terminal_all["service_date"].dt.month
terminal_all["day_of_week"] = terminal_all["service_date"].dt.day_name()

print(f"\nTotal runs: {len(terminal_all):,}")

2017: 2,315,468 pings -> 239,374 runs


2018: 1,480,001 pings -> 139,658 runs


2019: 1,990,712 pings -> 190,890 runs


2020: 1,121,532 pings -> 112,169 runs


2021: 1,580,251 pings -> 143,707 runs


2022: 1,859,407 pings -> 175,688 runs


2023: 2,871,273 pings -> 176,153 runs


2024: 3,428,108 pings -> 187,613 runs


2025: 3,484,652 pings -> 195,143 runs



Total runs: 1,560,395


In [4]:
# line key for later - ridership table's line names don't match line as-is
terminal_all["line_key"] = terminal_all["line"].map(line_key)

# date + scheduled departure datetime
# used later for weather merge and lateness-lag features
terminal_all["_pseudo_dt"] = (
    terminal_all["service_date"] + pd.to_timedelta(terminal_all["sched_origin_sec"], unit = "s")
)
n_missing_sched = terminal_all["_pseudo_dt"].isna().sum()
print(f"rows with unresolved sched_origin_sec (no 'departure' timestamp): {n_missing_sched:,}")

# localize to Eastern so asof-merge lines up w/ weather table
# ambiguous fall-back-DST times go to NA
# nonexistent spring-forward times shift forward
terminal_all["_sched_departure_dt"] = terminal_all["_pseudo_dt"].dt.tz_localize(
    eastern, ambiguous = "NaT", nonexistent = "shift_forward"
)
n_dst_ambiguous = terminal_all["_sched_departure_dt"].isna().sum() - n_missing_sched
print(f"additional rows with ambiguous DST departure time -> NaT: {n_dst_ambiguous:,}")

rows with unresolved sched_origin_sec (no 'departure' timestamp): 10,083
additional rows with ambiguous DST departure time -> NaT: 4


## 2. Peak/sports overlap against the scheduled window

Overlap against `sched_origin_sec`/`sched_terminus_sec`. Game windows are built from `5_home_games.parquet` (external, unchanged scraped historical schedule).

In [5]:
home_games = pd.read_parquet(f"{BASEPATH}/5_home_games.parquet")

game_duration = {"Phillies": 3, "Flyers": 2.5, "Eagles": 3.5, "Sixers": 2.5}

def hhmm_to_sec(t):
    h, m = t.split(":")
    return int(h) * 3600 + int(m) * 60

home_games["game_start_sec"] = home_games["game_time"].apply(hhmm_to_sec)
home_games["game_end_sec"] = (
    home_games["game_start_sec"] + home_games["team"].map(game_duration) * 3600
)
# pre/post windows around both start and end of the game
# catches arrival surge around kickoff 
# and departure surge around final whistle
home_games["pre_start_sec"] = home_games["game_start_sec"] - 90 * 60
home_games["post_start_sec"] = home_games["game_start_sec"] + 45 * 60
home_games["pre_end_sec"] = home_games["game_end_sec"] - 45 * 60
home_games["post_end_sec"] = home_games["game_end_sec"] + 90 * 60
home_games = home_games.rename(columns = {"game_date": "service_date"})

games = home_games[["service_date", "pre_start_sec", "post_start_sec", "pre_end_sec", "post_end_sec"]].copy()
print(f"{len(games):,} games with windows built")

1,372 games with windows built


In [6]:
overlap = terminal_all[RUN_COLS + ["sched_origin_sec", "sched_terminus_sec"]].merge(
    games, on = "service_date", how = "left"
)
# seconds of this run's scheduled window that fall inside
# the pre/post-start or pre/post-end game window
overlap["overlap_start_window_sec"] = (
    np.minimum(overlap["sched_terminus_sec"], overlap["post_start_sec"])
    - np.maximum(overlap["sched_origin_sec"], overlap["pre_start_sec"])
).clip(lower = 0)
overlap["overlap_end_window_sec"] = (
    np.minimum(overlap["sched_terminus_sec"], overlap["post_end_sec"])
    - np.maximum(overlap["sched_origin_sec"], overlap["pre_end_sec"])
).clip(lower = 0)
overlap["overlap_sec"] = overlap["overlap_start_window_sec"] + overlap["overlap_end_window_sec"]

# sum across all of a day's games 
# double/triple-header days can stack overlap from all games
sports_overlap = overlap.groupby(RUN_COLS).agg(sports_overlap_sec = ("overlap_sec", "sum")).reset_index()
terminal_all = terminal_all.merge(sports_overlap, on = RUN_COLS, how = "left")
terminal_all["sports_overlap_sec"] = terminal_all["sports_overlap_sec"].fillna(0)

del overlap, sports_overlap
gc.collect()
print("sports_overlap_sec summary:")
print(terminal_all["sports_overlap_sec"].describe())

sports_overlap_sec summary:
count    1.560395e+06
mean     2.851299e+02
std      9.448633e+02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.458000e+04
Name: sports_overlap_sec, dtype: float64


In [7]:
# same logic as sports against fixed AM/PM/midday windows instead of game windows
# AM/PM peaks derived empirically from observed data
# midday (11:00-15:00) added after 09e_error_analysis.ipynb found Airport-line
# misses clustering in this exact window
PEAK_WINDOWS_SEC = {
    "am_peak": (6 * 3600, 9 * 3600),
    "pm_peak": (15 * 3600, 18 * 3600),
    "midday": (11 * 3600, 15 * 3600),
}
for window_name, (win_start, win_end) in PEAK_WINDOWS_SEC.items():
    terminal_all[f"{window_name}_overlap_sec"] = (
        np.minimum(terminal_all["sched_terminus_sec"], win_end)
        - np.maximum(terminal_all["sched_origin_sec"], win_start)
    ).clip(lower = 0).fillna(0)

# bin the continuous overlap seconds into coarse buckets
# to create interaction term + allow for easier tree splits in pred
sec_bins = [-1, 0, 900, 1800, 3600, np.inf]
sec_labels = ["none", "0-15min", "15-30min", "30-60min", "60min+"]

terminal_all["sports_overlap_bin"] = pd.cut(terminal_all["sports_overlap_sec"], 
                                            bins = sec_bins, 
                                            labels = sec_labels)
terminal_all["am_peak_overlap_bin"] = pd.cut(terminal_all["am_peak_overlap_sec"], 
                                             bins = sec_bins, 
                                             labels = sec_labels)
terminal_all["pm_peak_overlap_bin"] = pd.cut(terminal_all["pm_peak_overlap_sec"], 
                                             bins = sec_bins, 
                                             labels = sec_labels)
terminal_all["midday_overlap_bin"] = pd.cut(terminal_all["midday_overlap_sec"],
                                            bins = sec_bins,
                                            labels = sec_labels)

# explicit interaction term
# PM peak overlapping a game letting out at the same time
# "none_x_none" forced to the front so it's the reference category 
# no AM interaction feature bc close to 0 games in morning -- too sparse
terminal_all["pm_peak_x_sports"] = (
    terminal_all["pm_peak_overlap_bin"].astype(str) + "_x_" + 
    terminal_all["sports_overlap_bin"].astype(str)
)
terminal_all["pm_peak_x_sports"] = pd.Categorical(
    terminal_all["pm_peak_x_sports"],
    categories = (["none_x_none"] + 
    [c for c in terminal_all["pm_peak_x_sports"].unique() if c != "none_x_none"])
)
print(terminal_all["pm_peak_x_sports"].value_counts())
print("\nmidday_overlap_bin value counts:")
print(terminal_all["midday_overlap_bin"].value_counts())

pm_peak_x_sports
none_x_none            1063949
30-60min_x_none         155897
15-30min_x_none          65983
60min+_x_none            57452
none_x_30-60min          53090
0-15min_x_none           34957
none_x_15-30min          28182
none_x_60min+            20539
none_x_0-15min           16028
30-60min_x_30-60min      14012
30-60min_x_15-30min       6798
30-60min_x_0-15min        5661
0-15min_x_30-60min        5008
15-30min_x_15-30min       4172
30-60min_x_60min+         3778
15-30min_x_30-60min       3285
60min+_x_30-60min         3139
0-15min_x_60min+          3056
60min+_x_0-15min          2895
60min+_x_60min+           2509
15-30min_x_60min+         2414
15-30min_x_0-15min        2408
60min+_x_15-30min         2362
0-15min_x_15-30min        1961
0-15min_x_0-15min          860
Name: count, dtype: int64

midday_overlap_bin value counts:
midday_overlap_bin
none        1211299
30-60min     174422
15-30min      78880
60min+        62682
0-15min       33112
Name: count, dtype: int64


## 3. Weather (from `06`'s prepped table), anchored to departure time 

In [8]:
wdf_aug = pd.read_parquet(f"{BASEPATH}/6_weather_prepped.parquet")
weather_cols = [c for c in wdf_aug.columns if c != "DATE"]

merge_input = (
    terminal_all[RUN_COLS + ["_sched_departure_dt"]]
    .dropna(subset = ["_sched_departure_dt"])
    .sort_values("_sched_departure_dt")
    .copy()
)
merge_input["_sched_departure_dt"] = merge_input["_sched_departure_dt"].astype("datetime64[ns, America/New_York]")

# asof-merge each run's scheduled departure against the weather table,
# trailing with direction="backward" so it only ever grabs weather at/before
# sched_origin, to prevent leakage (conditions after scheduled departure)
weather_features = pd.merge_asof(
    merge_input, wdf_aug,
    left_on = "_sched_departure_dt", right_on = "DATE",
    direction = "backward",
)
print(f"Matched weather for {len(weather_features):,} / {len(terminal_all):,} rows")

terminal_all = terminal_all.merge(weather_features[RUN_COLS + weather_cols], 
                                  on = RUN_COLS, 
                                  how = "left")
del wdf_aug, merge_input, weather_features
gc.collect()
print(f"terminal_all after weather merge: {terminal_all.shape}")

Matched weather for 1,550,308 / 1,560,395 rows


terminal_all after weather merge: (1560395, 78)


## 4. Schedule-derived per-run features

In [9]:
terminal_all = terminal_all.sort_values(["service_date", "line", 
                                         "direction_id", "sched_origin_sec"])

# minutes since the previous scheduled departure for same day/line/direction
# (NA for first run of day)
terminal_all["sched_headway_prior_min"] = (
    terminal_all
    .groupby(["service_date", "line", "direction_id"])["sched_origin_sec"].diff() / 60
)
# minutes until the next scheduled departure for same day/line/direction
# diff(periods=-1) looks forward. * -1 to keep positive
terminal_all["sched_headway_next_min"] = (
    -1 * terminal_all.groupby(["service_date", "line", "direction_id"])["sched_origin_sec"].diff(periods = -1)
) / 60

# where this run's stop count falls relative to other 
# runs with same origin/terminus pair
# separates express vs. local service, normalized w/in route
terminal_all["n_scheduled_stops_pctile_route"] = (
    terminal_all
    .groupby(["line", "origin_stop_id", "terminus_stop_id"])["n_scheduled_stops"].rank(pct = True)
)

# direction_id's 0/1 coding isn't consistent across lines, so
# infer which direction_id is "outbound" per line based on 
# which one originates in the Center City trunk stations more often
TRUNK_STOP_IDS = {"90004", "90005", "90006", "90007", "90008", "90009"}
line_direction_outbound_share = (
    terminal_all.assign(_origin_in_trunk = terminal_all["origin_stop_id"].isin(TRUNK_STOP_IDS))
    .groupby(["line", "direction_id"])["_origin_in_trunk"]
    .mean()
)
outbound_direction_by_line = line_direction_outbound_share.unstack("direction_id").idxmax(axis = 1)

# flip to a consistent inbound/outbound flag
# using mapping from above
# if any unresolved direction_id, leave NA
terminal_all["is_inbound"] = (
    terminal_all["direction_id"] != terminal_all["line"].map(outbound_direction_by_line)
).astype("boolean")
terminal_all.loc[terminal_all["direction_id"].isna(), "is_inbound"] = pd.NA

print("is_inbound mean:", terminal_all["is_inbound"].mean())

is_inbound mean: 0.5157582229073079


## 5. Expanded lagged lateness features
At train, line, system level; across varying periods

**Same line-direction-day features**
Looking @ the prior run from the same line/direction/day, relative to "current" train run:
- `lag_line_lateness_min1` — prior run's own terminal lateness (min)
- `lag_line_lateness_std` — prior run's volatility (min) - what was the std in min across each of its feed pings during its run
- `lag_line_lateness_mean_run` — prior run's average lateness (min) - what was the avg of each of its feed pings during its run


In [10]:
# lateness of the immediately preceding run on the same day/line/direction
terminal_all["lateness_prior_run_min"] = terminal_all.groupby(
    ["service_date", "line", "direction_id"]
)["lateness"].shift(1)
terminal_all.loc[terminal_all["sched_origin_sec"].isna(), "lateness_prior_run_min"] = np.nan

# volatility of the prior run's lateness readings
# na for the first run of a day (no prior run)
# or when the prior run itself had only one observation
terminal_all["lag_line_lateness_std"] = terminal_all.groupby(
    ["service_date", "line", "direction_id"]
)["lateness_std"].shift(1)
terminal_all.loc[terminal_all["sched_origin_sec"].isna(), "lag_line_lateness_std"] = np.nan
terminal_all = terminal_all.drop(columns = ["lateness_std"])  # current run's own value is never a valid feature

# average lateness across the prior run's readings
# first runs get fallback to trailing average (below)
terminal_all["lateness_mean_run_prior"] = terminal_all.groupby(
    ["service_date", "line", "direction_id"]
)["lateness_mean_run"].shift(1)
terminal_all.loc[terminal_all["sched_origin_sec"].isna(), "lateness_mean_run_prior"] = np.nan

# first/last run of the day for this line/direction flag.
# na (not False) when schedule is unresolved
terminal_all["is_first_run_of_day"] = (
    terminal_all.groupby(["service_date", "line", "direction_id"]).cumcount() == 0
).astype("boolean")
terminal_all.loc[terminal_all["sched_origin_sec"].isna(), "is_first_run_of_day"] = pd.NA

terminal_all["is_last_run_of_day"] = (
    terminal_all.groupby(["service_date", "line", "direction_id"]).cumcount(ascending = False) == 0
).astype("boolean")
terminal_all.loc[terminal_all["sched_origin_sec"].isna(), "is_last_run_of_day"] = pd.NA

# fallback when no prior run to lag for a day/line/direction
# trailing 4-week average of that day of week/line/direction's first run lateness
# so earliest trains still can get a signal
first_runs = (
    terminal_all.loc[
        terminal_all["is_first_run_of_day"].fillna(False),
        ["service_date", "line", "direction_id", "lateness", "lateness_mean_run"],
    ]
    .rename(columns = {"lateness": "_first_run_lateness", "lateness_mean_run": "_first_run_mean_lateness"})
    .copy()
)
first_runs["_dow"] = first_runs["service_date"].dt.dayofweek
first_runs = first_runs.sort_values(["line", "direction_id", "_dow", "service_date"])
first_runs["lateness_first_run_trailing4_min"] = (
    first_runs.groupby(["line", "direction_id", "_dow"])["_first_run_lateness"]
    .transform(lambda s: s.shift(1).rolling(window = 4, min_periods = 1).mean())
)
first_runs["lateness_first_run_mean_trailing4_min"] = (
    first_runs.groupby(["line", "direction_id", "_dow"])["_first_run_mean_lateness"]
    .transform(lambda s: s.shift(1).rolling(window = 4, min_periods = 1).mean())
)
terminal_all = terminal_all.merge(
    first_runs[[
        "service_date", "line", "direction_id",
        "lateness_first_run_trailing4_min", "lateness_first_run_mean_trailing4_min",
    ]],
    on = ["service_date", "line", "direction_id"], how = "left",
)
# combine: prior-run lag where it exists, first-run trailing average elsewhere 
# keep only combined columns
terminal_all["lag_line_lateness_min"] = terminal_all["lateness_prior_run_min"].fillna(
    terminal_all["lateness_first_run_trailing4_min"]
)
terminal_all["lag_line_lateness_mean_run"] = terminal_all["lateness_mean_run_prior"].fillna(
    terminal_all["lateness_first_run_mean_trailing4_min"]
)
terminal_all = terminal_all.drop(columns = [
    "lateness_prior_run_min", "lateness_first_run_trailing4_min",
    "lateness_mean_run_prior", "lateness_mean_run", "lateness_first_run_mean_trailing4_min",
])
del first_runs
# print("lag_line_lateness_min built")
# print("lag_line_lateness_mean_run summary:")
# print(terminal_all["lag_line_lateness_mean_run"].describe())
# print("lag_line_lateness_std summary:")
# print(terminal_all["lag_line_lateness_std"].describe())
# print("is_first_run_of_day mean:", terminal_all["is_first_run_of_day"].mean())
# print("is_last_run_of_day mean:", terminal_all["is_last_run_of_day"].mean())

**Network lateness**
Looking @ entire network on same day as "current" train run
- `lag_network_lateness_30min_mean` — avg lateness across every SEPTA run in the last 30 minutes before the run's departure
- `lag_network_lateness_2hr_mean` — same, 2 hours
- `lag_network_lateness_6hr_mean` — same, 6 hours


In [11]:
# network-wide lag needs a chronological ordering across all lines
# vs. per-line groups
# rebuild pseudo-departure times and sort on them
resolved_mask = terminal_all["sched_origin_sec"].notna()
resolved = terminal_all.loc[resolved_mask, ["service_date", "sched_origin_sec", "lateness"]].copy()
resolved["_pseudo_dt"] = resolved["service_date"] + pd.to_timedelta(resolved["sched_origin_sec"], unit = "s")
resolved = resolved.sort_values(["service_date", "_pseudo_dt"])
sorted_index = resolved.index

# trailing average lateness across the whole network in the X minutes
# before this run's scheduled departure
# closed = "left" excludes the run itself to prevent leakage.
# since we don't have SEPTA system alerts/disruption alerts,
# this should help capture some system-wide disruptions
NETWORK_LAG_WINDOWS_MIN = {"30min": 30, "2hr": 120, "6hr": 360}
for name, X in NETWORK_LAG_WINDOWS_MIN.items():
    network_roll = (
        resolved.reset_index(drop = True)
        .groupby("service_date")
        .rolling(f"{X}min", on = "_pseudo_dt", closed = "left")["lateness"]
        .agg(["mean"])
        .reset_index(drop = True)
    )
    network_lag_series = pd.Series(network_roll["mean"].values, index = sorted_index)
    terminal_all[f"lag_network_lateness_{name}_mean"] = network_lag_series.reindex(terminal_all.index)

del resolved
print("network lag features built:", list(NETWORK_LAG_WINDOWS_MIN.keys()))

network lag features built: ['30min', '2hr', '6hr']


**Line-direction lateness**
Looking @ same line / direction as current train
- `lag_line_lateness_1d_mean` — average lateness of all trains yesterday (1 day trailing window)
- `lag_line_lateness_7d_mean` — same, 1 week
- `lag_line_lateness_30d_mean` — same, 1 month

In [12]:
# collapse to one row per line/direction/day before rolling
# these trends operate daily not per run
daily_line_lateness = (
    terminal_all.groupby(["line", "direction_id", "service_date"])["lateness"]
    .mean().reset_index().sort_values(["line", "direction_id", "service_date"])
)

# trailing X-day average of that line/direction's daily mean lateness,
# closed="left" to prevent leak from "today"
# short vs. long window separates a recent bad day from persistent slowness
LINE_TREND_WINDOWS_DAYS = {"1d": 1, "7d": 7, "30d": 30}
for name, X in LINE_TREND_WINDOWS_DAYS.items():
    daily_roll = (
        daily_line_lateness.groupby(["line", "direction_id"])
        .rolling(f"{X}D", on = "service_date", closed = "left")["lateness"]
        .mean().reset_index(drop = True)
    )
    daily_line_lateness[f"lag_line_lateness_{name}_mean"] = daily_roll.values

merge_cols = (["line", "direction_id", "service_date"] + 
              [f"lag_line_lateness_{n}_mean" for n in LINE_TREND_WINDOWS_DAYS])
terminal_all = terminal_all.merge(daily_line_lateness[merge_cols], 
                                  on = ["line", "direction_id", "service_date"], 
                                  how = "left")

# null out for runs with unresolved direction_id 
# to keep them from groupin together + signaling incorrectly
for name in LINE_TREND_WINDOWS_DAYS:
    terminal_all.loc[terminal_all["direction_id"].isna(), f"lag_line_lateness_{name}_mean"] = np.nan

del daily_line_lateness
print("line trend lag features built:", list(LINE_TREND_WINDOWS_DAYS.keys()))

line trend lag features built: ['1d', '7d', '30d']


**Physical train features**

Looking @ the same physical train's own trailing lateness, grouped by train_number (so it will folow train even across a line change, unlike line-level lags)
- `lag_train_lateness_1run_mean` — the physical train's prior run raw lateness.
- `lag_train_lateness_3run_mean` — average lateness over its last 3 runs.
- `lag_train_lateness_5run_mean` — average over its last 5 runs.
- `lag_train_lateness_10run_mean` — average over its last 10 runs.

One non-lagged feature:
- `sched_turnaround_gap_min` — scheduled minutes betwen physical train's previous scheduled arrival and its "current" scheduled departure. this follows train, not line, to accommodate through-running trains (Which would likely have a very low value here since they go immediately to next run outbound) 

In [ ]:
terminal_all = terminal_all.sort_values(["train_number", "service_date", "sched_origin_sec"])
# scheduled turnaround gap for this physical train
_pseudo_departure_dt = terminal_all["service_date"] + pd.to_timedelta(terminal_all["sched_origin_sec"], unit = "s")
_pseudo_arrival_dt = terminal_all["service_date"] + pd.to_timedelta(terminal_all["sched_terminus_sec"], unit = "s")
_prev_arrival_dt = _pseudo_arrival_dt.groupby(terminal_all["train_number"]).shift(1)
terminal_all["sched_turnaround_gap_min"] = (_pseudo_departure_dt - _prev_arrival_dt).dt.total_seconds() / 60

# trailing average lateness of this specific physical train over last N runs
# regardless of line - might help catch individual train issues
# 1run = the immediately preceding run's raw lateness; for a multi-line run 
# continuation this is the first leg's terminal lateness so it carries across line change
TRAIN_LAG_WINDOWS_RUNS = {"1run": 1, "3run": 3, "5run": 5, "10run": 10}
for name, N in TRAIN_LAG_WINDOWS_RUNS.items():
    terminal_all[f"lag_train_lateness_{name}_mean"] = (
        terminal_all.groupby("train_number")["lateness"]
        .transform(lambda s: s.shift(1).rolling(window = N, min_periods = 1).mean())
    )

print("train lag features built:", list(TRAIN_LAG_WINDOWS_RUNS.keys()))
print("sched_turnaround_gap_min summary:")
print(terminal_all["sched_turnaround_gap_min"].describe())
print(f"terminal_all shape after all lag features: {terminal_all.shape}")

train lag features built: ['1run', '3run', '5run', '10run']
sched_turnaround_gap_min summary:
count    1.543706e+06
mean     2.200648e+03
std      3.408239e+04
min     -1.200000e+02
25%      0.000000e+00
50%      1.346000e+03
75%      1.386000e+03
max      4.286956e+06
Name: sched_turnaround_gap_min, dtype: float64
terminal_all shape after all lag features: (1560395, 96)


## 6. Deferred merges

Ridership and calendar. Calendar exceptions come from two tables: `5_calendar_by_date.parquet`
(system-wide exceptions, merged by `service_date` alone) and `5_calendar_by_date_line.parquet`
(exceptions that resolve to specific line(s) via the GTFS crosswalk, merged by `(service_date, line)`) in `05_calendar_events.ipynb`.
The two are summed into the same final `service_additions`/`service_removals` columns so nothing downstream needs to change shape.

In [14]:
calendar_by_date = pd.read_parquet(f"{BASEPATH}/5_calendar_by_date.parquet")
calendar_by_date_line = pd.read_parquet(f"{BASEPATH}/5_calendar_by_date_line.parquet")

# day-before/day-after holiday flags
# to account for potentially different ridership patterns in holiday windows
holiday_dates = calendar_by_date.loc[calendar_by_date["is_holiday"] == 1, "service_date"]
day_before_holiday = (
    pd.DataFrame({"service_date": holiday_dates - pd.Timedelta(days = 1), "is_day_before_holiday": 1})
    .drop_duplicates(subset = ["service_date"])
)
day_after_holiday = (
    pd.DataFrame({"service_date": holiday_dates + pd.Timedelta(days = 1), "is_day_after_holiday": 1})
    .drop_duplicates(subset = ["service_date"])
)

# leak-free replacement for ridership table (built in 04)
# already keyed on line_key and pre-joined to each row's last completed month
# so plain merge here
ridership_monthly = pd.read_parquet(f"{BASEPATH}/4_ridership_by_line_month_safe.parquet")

terminal_all = (
    terminal_all
    .merge(calendar_by_date, on = "service_date", how = "left")
    .merge(calendar_by_date_line, on = ["service_date", "line"], how = "left")
    .merge(day_before_holiday, on = "service_date", how = "left")
    .merge(day_after_holiday, on = "service_date", how = "left")
    .merge(
        ridership_monthly[["line_key", "year", "month", "avg_daily_boards_monthly"]],
        on = ["line_key", "year", "month"], how = "left",
    )
    .drop(columns = ["line_key"])
)

# missing calendar match means no special service event that day (or, for the
# line-specific columns, no exception affecting *this* line specifically) --
# fill to 0, then combine system-wide (05's broad bucket) + line-specific
# (05's specific bucket) into one final count per run
terminal_all["service_additions"] = (
    terminal_all["service_additions"].fillna(0) + terminal_all["service_additions_line"].fillna(0)
).astype(int)
terminal_all["service_removals"] = (
    terminal_all["service_removals"].fillna(0) + terminal_all["service_removals_line"].fillna(0)
).astype(int)
terminal_all = terminal_all.drop(columns = ["service_additions_line", "service_removals_line"])
terminal_all["any_service_exception"] = (
    (terminal_all["service_additions"] > 0) | (terminal_all["service_removals"] > 0)
).astype(int)

terminal_all["is_holiday"] = terminal_all["is_holiday"].fillna(0).astype(int)
terminal_all["is_day_before_holiday"] = terminal_all["is_day_before_holiday"].fillna(0).astype(int)
terminal_all["is_day_after_holiday"] = terminal_all["is_day_after_holiday"].fillna(0).astype(int)

print(f"terminal_all after deferred merges: {terminal_all.shape}")
print(f"any_service_exception mean: {terminal_all['any_service_exception'].mean():.4f}")

terminal_all after deferred merges: (1560395, 102)
any_service_exception mean: 0.0190


## 6b. Spatial position (distance from Center City)

Adds `stops_from_cc`: how many stops out a run's origin is from the Center City trunk using most recent GTFS release's stop order per line/direction. 

0 = at the trunk for both directions so inbound (flipped) and outbound are on same scale.

~94% match rate. Some stop ids drifted since 2017 and don't match this table from most recent release (mostly Warminster, Media/Wawa, West Trenton, Lansdale/Doylestown) — left NA since stops are generally static otherwise and not concerned about data quality otherwise

Used in `11_spatial_delay_patterns.ipynb` later

In [ ]:
import geopandas as gpd

CC_TRUNK_STOPS = ["90004", "90005", "90006", "90007", "90008", "90009"]
PROJECTED_CRS = "EPSG:32618"

# station locations, projected so distance can be measured in meters
stn_rider = pd.read_csv(f"{BASEPATH}/4_ridership_by_station.csv")
stn_rider["stop_id"] = stn_rider["stop_id"].astype(str)
stn_points = gpd.GeoDataFrame(
    stn_rider.drop_duplicates("stop_id").copy(),
    geometry = gpd.points_from_xy(
        stn_rider.drop_duplicates("stop_id")["longitude"],
        stn_rider.drop_duplicates("stop_id")["latitude"],
    ),
    crs = "EPSG:4326",
).to_crs(PROJECTED_CRS).set_index("stop_id")

cc_pts = stn_points.loc[stn_points.index.intersection(CC_TRUNK_STOPS)]
cc_centroid_x = cc_pts.geometry.x.mean()
cc_centroid_y = cc_pts.geometry.y.mean()

# most recent GTFS release's stop order for each line/direction
crosswalk_recent = pd.read_parquet(
    f"{BASEPATH}/2_gtfs_linkages_since_2017_clean.parquet"
)
crosswalk_recent["line"] = crosswalk_recent["line"].str.replace(
    " Line$", "", regex = True
)
crosswalk_recent = crosswalk_recent.rename(
    columns = {"gtfs_date": "source_gtfs_date"}
)

stop_times_all = pd.read_parquet(f"{BASEPATH}/2_gtfs_stop_times.parquet")
stop_times_all["stop_sequence"] = stop_times_all["stop_sequence"].astype(int)
stop_times_all = stop_times_all.rename(
    columns = {"gtfs_date": "source_gtfs_date"}
)

recent_date = crosswalk_recent["source_gtfs_date"].max()
recent_cols = ["line", "trip_id", "source_gtfs_date", "direction_id"]
recent_trips = (
    crosswalk_recent[crosswalk_recent["source_gtfs_date"] == recent_date]
    [recent_cols]
    .drop_duplicates()
    .merge(
        stop_times_all, on = ["trip_id", "source_gtfs_date"], how = "inner"
    )
)

# longest trip per line/direction = the one that hits every stop
trip_lengths = (
    recent_trips
    .groupby(["line", "trip_id", "direction_id"])["stop_sequence"]
    .count()
    .reset_index(name = "n_stops")
)
best_trip_per_dir = trip_lengths.sort_values(
    "n_stops", ascending = False
).drop_duplicates(["line", "direction_id"])

# label a trip outbound/inbound by which end sits closer to Center City
# bc direction_id's 0/1 coding isn't consistent across lines
def orient_trip(trip_id):
    ordered = (
        recent_trips[recent_trips["trip_id"] == trip_id]
        .sort_values("stop_sequence")["stop_id"]
        .tolist()
    )
    ordered = [s for s in ordered if s in stn_points.index]
    if len(ordered) < 2:
        return None, None
    first_pt = stn_points.loc[ordered[0], "geometry"]
    last_pt = stn_points.loc[ordered[-1], "geometry"]
    dist_first = (
        (first_pt.x - cc_centroid_x) ** 2 + (first_pt.y - cc_centroid_y) ** 2
    )
    dist_last = (
        (last_pt.x - cc_centroid_x) ** 2 + (last_pt.y - cc_centroid_y) ** 2
    )
    return ("Outbound" if dist_first < dist_last else "Inbound"), ordered

line_dir_rows = []
for _, r in best_trip_per_dir.iterrows():
    real_direction, ordered = orient_trip(r["trip_id"])
    if ordered is None:
        continue
    line_dir_rows.append({
        "line": r["line"], "real_direction": real_direction,
        "ordered_stops": ordered,
    })
line_dirs = pd.DataFrame(line_dir_rows)

combo_check = line_dirs.groupby("line")["real_direction"].apply(
    lambda s: sorted(s.tolist())
)
expected = ["Inbound", "Outbound"]
n_bad = (combo_check != combo_check.apply(lambda x: expected)).sum()
print(f"{n_bad} lines not cleanly Outbound+Inbound (should be 0)")

# flatten to one row per (line, direction, stop), numbered in order
line_dir_stop_rows = []
for _, r in line_dirs.iterrows():
    for i, s in enumerate(r["ordered_stops"]):
        line_dir_stop_rows.append({
            "line": r["line"], "real_direction": r["real_direction"],
            "stop_position": i, "stop_id": s,
        })
stops = pd.DataFrame(line_dir_stop_rows)
del stn_rider, stn_points, crosswalk_recent, stop_times_all, recent_trips
del trip_lengths, best_trip_per_dir, line_dirs, line_dir_stop_rows

0 lines not cleanly Outbound+Inbound (should be 0)


In [16]:
# stop_position counts outward from each direction's own origin -- flip
# Inbound so 0 = at the Center City trunk for both directions
max_pos = stops.groupby(
    ["line", "real_direction"]
)["stop_position"].transform("max")
stops["stops_from_cc"] = np.where(
    stops["real_direction"] == "Outbound",
    stops["stop_position"],
    max_pos - stops["stop_position"],
)
stops_join = (
    stops[["line", "real_direction", "stop_id", "stops_from_cc"]]
    .drop_duplicates()
    .rename(columns = {
        "line": "_sj_line", "real_direction": "_sj_dir",
        "stop_id": "_sj_stop_id",
    })
)

In [ ]:
# this table only has "Media/Wawa" (current name); Media/Elwyn's
# stops are the same corridor before the extension, so join those rows
# against Wawa's stop position table directly
terminal_all["_join_line"] = terminal_all["line"].replace(
    {"Media/Elwyn": "Media/Wawa"}
)
terminal_all["_direction_key"] = np.where(
    terminal_all["is_inbound"].fillna(False), "Inbound", "Outbound"
)

terminal_all = terminal_all.merge(
    stops_join, left_on = ["_join_line", "_direction_key", "origin_stop_id"],
    right_on = ["_sj_line", "_sj_dir", "_sj_stop_id"], how = "left",
).drop(columns = [
    "_join_line", "_direction_key", "_sj_line", "_sj_dir", "_sj_stop_id",
])

del stops, stops_join
print(
    f"stops_from_cc match rate: "
    f"{terminal_all['stops_from_cc'].notna().mean():.2%}"
)
print(terminal_all["stops_from_cc"].describe())

stops_from_cc match rate: 93.83%
count    1.464067e+06
mean     8.417277e+00
std      8.269623e+00
min      0.000000e+00
25%      0.000000e+00
50%      9.000000e+00
75%      1.500000e+01
max      2.500000e+01
Name: stops_from_cc, dtype: float64


## 7. GTFS QC

In [18]:
n_before = len(terminal_all)
terminal_all = terminal_all[terminal_all["train_number"] != 0]
print(f"Dropped {n_before - len(terminal_all):,} rows with train_number == 0")

n_before = len(terminal_all)
terminal_all = terminal_all[terminal_all["line"].notna()]
print(f"Dropped {n_before - len(terminal_all):,} rows with unmatched/missing line")

n_before = len(terminal_all)
terminal_all = terminal_all[terminal_all["line"] != "Center City Only Trains"]
print(f"Dropped {n_before - len(terminal_all):,} rows with line == 'Center City Only Trains'")

neg_sched = terminal_all["sched_duration_sec"] < 0
terminal_all.loc[neg_sched, "sched_duration_sec"] = np.nan
print(f"Nulled {neg_sched.sum():,} rows with impossible negative sched_duration_sec")

# same treatment as sched_duration_sec above: neg turnaround gap means
# schedule has next departure before previous arrival, which is impossible
# ~0.6% of rows, set to na
neg_turnaround = terminal_all["sched_turnaround_gap_min"] < 0
terminal_all.loc[neg_turnaround, "sched_turnaround_gap_min"] = np.nan
print(f"Nulled {neg_turnaround.sum():,} rows with impossible negative sched_turnaround_gap_min")

print(f"\nFinal row count: {len(terminal_all):,}")

Dropped 1,472 rows with train_number == 0


Dropped 7,130 rows with unmatched/missing line


Dropped 39 rows with line == 'Center City Only Trains'
Nulled 11 rows with impossible negative sched_duration_sec
Nulled 9,051 rows with impossible negative sched_turnaround_gap_min

Final row count: 1,551,754


## 8. Weather patches

In [19]:
# missing gust/ice readings mean none was recorded, not unknown
# 0 is a real value here so any_ flags allow for separate binary signal
terminal_all["gust_ms_asof"] = terminal_all["gust_ms_asof"].fillna(0)
terminal_all["any_gust_asof"] = (terminal_all["gust_ms_asof"] > 0).astype(int)

terminal_all["ice_accretion_cm_asof"] = terminal_all["ice_accretion_cm_asof"].fillna(0)
terminal_all["any_ice_accretion_asof"] = (terminal_all["ice_accretion_cm_asof"] > 0).astype(int)

# same fill applied to the lag versions of these columns
for window in ["3h", "6h", "24h"]:
    for col_prefix in ["gust_ms", "ice_accretion_cm", "precip_mm"]:
        col = f"{col_prefix}_{window}"
        if col in terminal_all.columns:
            terminal_all[col] = terminal_all[col].fillna(0)

# cloud_base_m_1_asof NA means no cloud layer was reported (clear sky) 
# capture that as its own flag before fill
terminal_all["has_cloud_layer"] = terminal_all["cloud_base_m_1_asof"].notna().astype(int)
# -1 sentinel for missing/calm wind direction -- keeps it distinguishable from a real 0-360 degree reading
terminal_all["wind_dir_asof"] = terminal_all["wind_dir_asof"].fillna(-1)

# a station can report a precip code (rain/snow etc) w/o listing an amount
# flag these separately before defaulting missing to 0
# so recorded "no precip" and "precip happened but wasn't measured" 
# aren't confused
precip_na = terminal_all["precip_mm_asof"].isna()
precip_codes = ["SR", "MR", "HR", "DZ", "HDZ", "SS", "MS", "HS", "FR", "RS",
                "IP", "SP", "IC", "HL", "TSLR", "TSMR", "TSHY", "TSHL", "PU"]
suspect = precip_na & terminal_all["wx_primary_asof"].isin(precip_codes)
terminal_all["precip_unquantified"] = suspect.astype(int)
terminal_all.loc[precip_na, "precip_mm_asof"] = terminal_all.loc[precip_na, "precip_mm_asof"].fillna(0)
terminal_all.loc[precip_na, "precip_trace_asof"] = terminal_all.loc[precip_na, "precip_trace_asof"].fillna(False)
terminal_all["precip_trace_asof"] = terminal_all["precip_trace_asof"].astype(bool)

# missing readings here are actually missing
# median fill
terminal_all["temp_c_asof"] = terminal_all["temp_c_asof"].fillna(terminal_all["temp_c_asof"].median())
terminal_all["wind_speed_ms_asof"] = terminal_all["wind_speed_ms_asof"].fillna(terminal_all["wind_speed_ms_asof"].median())

#print("weather NaN fills applied")

In [20]:
# collapse rare weather codes so we don't end up with a bunch of low-n categories
RARE_THRESHOLD = 1000
wx_mode = terminal_all["wx_primary_asof"]
wx_counts = wx_mode.value_counts()
rare_codes = wx_counts[wx_counts < RARE_THRESHOLD].index.tolist()
print(f"Collapsing {len(rare_codes)} rare codes into 'Z_RARE'")

# "Z_RARE"/"A_CLR" prefixes sort the catch-all categories for reference
# rare categories "Z" -> last 
# clear sky (most common) "A" -> first in alphabetical prints
terminal_all["wx_primary_asof_collapsed"] = (
    wx_mode.astype(str)
    .where(~wx_mode.isin(rare_codes), "Z_RARE")
    .where(wx_mode.notna(), "A_CLR")
)
print(terminal_all["wx_primary_asof_collapsed"].value_counts())

Collapsing 18 rare codes into 'Z_RARE'


wx_primary_asof_collapsed
A_CLR     1360353
SR          90294
MI          37208
SS          12058
PU          11203
HR          10131
DZ           7069
MR           6693
DU           5215
HZ           4444
Z_RARE       3304
TS           2526
TSLR         1256
Name: count, dtype: int64


## 8. Save

In [21]:
terminal_all = terminal_all.drop(columns = ["_pseudo_dt", "_sched_departure_dt"])
terminal_all.to_parquet(f"{BASEPATH}/7_df_departure_ready.parquet")
print(f"Saved {len(terminal_all):,} rows, {terminal_all.shape[1]} columns to 7_df_departure_ready.parquet")

Saved 1,551,754 rows, 106 columns to 7_df_departure_ready.parquet
